# Bias mitigation

Bias mitigation changes data, thresholds, or losses so the selected model pays attention to fairness constraints.

Bias mitigation sits where prediction quality is no longer enough. We keep the classifier small and CPU-only, but evaluate fairness gaps, validation behavior, and calibration-like checks across D1-D5. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 19
rng = np.random.default_rng(SEED)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def make_group(X, y):
    scores = X[:, 0]
    if len(np.unique(scores)) < 3:
        scores = X.sum(axis=1)
    cutoff = np.median(scores)
    group = (scores > cutoff).astype(int)
    if len(np.unique(group)) < 2:
        group = (np.arange(len(y)) % 2).astype(int)
    return group


def safe_split(X, y, group=None, test_size=0.4):
    stratify = y
    if min(np.bincount(y.astype(int))) < 2:
        stratify = None
    pieces = train_test_split(
        X,
        y,
        np.arange(len(y)),
        test_size=test_size,
        random_state=0,
        stratify=stratify,
    )
    x_train, x_test, y_train, y_test, idx_train, idx_test = pieces
    if group is None:
        return x_train, x_test, y_train, y_test, idx_train, idx_test, None, None
    return x_train, x_test, y_train, y_test, idx_train, idx_test, group[idx_train], group[idx_test]


def fit_scaled_logreg(X, y, C=1.0):
    x_train, x_test, y_train, y_test, idx_train, idx_test, _, _ = safe_split(X, y)
    scaler = StandardScaler()
    x_train_s = scaler.fit_transform(x_train)
    x_test_s = scaler.transform(x_test)
    model = LogisticRegression(max_iter=2000, C=C, multi_class="auto")
    model.fit(x_train_s, y_train)
    return model, scaler, x_train_s, x_test_s, y_train, y_test, idx_train, idx_test


def probability_for_label(model, X, y):
    probs = model.predict_proba(X)
    positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    return probs[np.arange(len(y)), positions]


def fgsm_attack(model, X, y, eps):
    probs = model.predict_proba(X)
    class_positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(y)), class_positions] = 1.0
    grad = (probs - one_hot) @ model.coef_
    direction = np.sign(grad)
    return X + eps * direction


def robust_accuracy_for_eps(model, X, y, eps):
    attacked = fgsm_attack(model, X, y, eps)
    preds = model.predict(attacked)
    return accuracy_score(y, preds)


def fairness_report(y, yhat, group):
    rows = {}
    positive = int(np.max(y))
    for g in [0, 1]:
        mask = group == g
        truth_pos = mask & (y == positive)
        truth_neg = mask & (y != positive)
        pred_pos = mask & (yhat == positive)
        tp = int(np.sum(pred_pos & truth_pos))
        fp = int(np.sum(pred_pos & truth_neg))
        fn = int(np.sum((~pred_pos) & truth_pos))
        tn = int(np.sum((~pred_pos) & truth_neg))
        rate = float(np.mean(yhat[mask] == positive)) if np.any(mask) else np.nan
        tpr = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)
        rows[g] = {"n": int(mask.sum()), "pos_rate": rate, "tpr": tpr, "fpr": fpr, "tp": tp, "fp": fp, "fn": fn, "tn": tn}
    dp_gap = abs(rows[0]["pos_rate"] - rows[1]["pos_rate"])
    eo_gap = max(abs(rows[0]["tpr"] - rows[1]["tpr"]), abs(rows[0]["fpr"] - rows[1]["fpr"]))
    return {"group0": rows[0], "group1": rows[1], "dp_gap": dp_gap, "eo_gap": eo_gap}


def plot_2d_projection(ax, X, color, title):
    if X.shape[1] >= 2:
        shown = X[:, :2]
    else:
        shown = np.column_stack([X[:, 0], np.zeros(len(X))])
    ax.scatter(shown[:, 0], shown[:, 1], c=color, s=18, cmap="viridis", alpha=0.75)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once: fairness-regularized mitigation

The lesson objective is $$J(\theta)=R_S(\theta)+\lambda\,\Delta(\theta).$$ We first verify the plan's worked values, then reuse threshold post-processing as the concrete mitigation.

In [ ]:

components = np.array([0.35, 0.12, 0.18], dtype=float)
knob = 0.5
total = float(np.sum(components))
absolute_total = float(np.sum(np.abs(components)))
leading_share = float(abs(components[0]) / absolute_total)
guarded = float(total + knob * absolute_total)
contrast = float(total - components[2])
change = float(contrast - total)

assert np.isclose(total, 0.65)
assert np.isclose(absolute_total, 0.65)
assert np.isclose(round(guarded, 3), 0.975)
assert np.isclose(round(leading_share, 3), 0.538)
assert np.isclose(knob, 0.500)

print("total", round(total, 3))
print("absolute_total", round(absolute_total, 3))
print("leading_share", round(leading_share, 3))
print("guarded", round(guarded, 3))
print("change_without_component_3", round(change, 3))


A practical mitigation can scan thresholds and keep the candidate with the smallest validation objective: empirical error plus $\lambda$ times the demographic-parity gap.

In [ ]:

def mitigate_bias(model, X_valid, y_valid, group_valid, lambda_=0.5, method="threshold"):
    positive = int(np.max(y_valid))
    probabilities = model.predict_proba(X_valid)
    pos_index = np.where(model.classes_ == positive)[0][0]
    scores = probabilities[:, pos_index]
    best = None
    thresholds = np.linspace(0.2, 0.8, 25)
    for threshold in thresholds:
        preds = np.where(scores >= threshold, positive, int(np.min(y_valid)))
        report = fairness_report(y_valid, preds, group_valid)
        error = 1.0 - accuracy_score(y_valid, preds)
        objective = error + lambda_ * report["dp_gap"]
        candidate = {"threshold": threshold, "preds": preds, "error": error, "report": report, "objective": objective}
        if best is None or objective < best["objective"]:
            best = candidate
    return best

X, y = clf_ladder()[0][1:]
group = make_group(X, y)
model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
best = mitigate_bias(model, x_test, y_test, group[idx_test], lambda_=0.5)
print("chosen_threshold", round(best["threshold"], 3))
print("validation_objective", round(best["objective"], 3))
print("dp_gap", round(best["report"]["dp_gap"], 3))


## The dataset ladder

The same classifier family is tested on D1-D5: a hand toy, synthetic blobs, noisy moons, real Wine data, and real Breast Cancer data.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    print(name)
    print("  shape", X.shape)
    print("  classes", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same method across D1-D5

The metric is fairness gap at constrained accuracy: among thresholds within five points of clean accuracy, choose the lowest demographic-parity gap.

In [ ]:

mitigation_results = []
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    group = make_group(X, y)
    split = safe_split(X, y, group)
    x_train, x_test, y_train, y_test, idx_train, idx_test, g_train, g_test = split
    scaler = StandardScaler()
    x_train_s = scaler.fit_transform(x_train)
    x_test_s = scaler.transform(x_test)
    model = LogisticRegression(max_iter=2000, multi_class="auto")
    model.fit(x_train_s, y_train)
    clean_preds = model.predict(x_test_s)
    clean_acc = accuracy_score(y_test, clean_preds)
    clean_report = fairness_report(y_test, clean_preds, g_test)
    best = mitigate_bias(model, x_test_s, y_test, g_test, lambda_=0.5)
    mitigated_acc = 1.0 - best["error"]
    mitigation_results.append({"rung": rung, "name": name, "clean_acc": clean_acc, "clean_gap": clean_report["dp_gap"], "mitigated_acc": mitigated_acc, "mitigated_gap": best["report"]["dp_gap"], "threshold": best["threshold"]})

print("rung | clean_acc | clean_gap | mitigated_acc | mitigated_gap | threshold")
for row in mitigation_results:
    print(row["rung"], round(row["clean_acc"], 3), round(row["clean_gap"], 3), round(row["mitigated_acc"], 3), round(row["mitigated_gap"], 3), round(row["threshold"], 3))


## Results visualization

Left: per-rung clean versus mitigated accuracy/gap dashboards. Right: fairness-gap curve across ladder stress.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, row in zip(axes[0], mitigation_results):
    ax.bar(["clean", "mitigated"], [row["clean_acc"], row["mitigated_acc"]], color=["tab:blue", "tab:green"])
    ax.set_ylim(0, 1.05)
    ax.set_title(f"D{row['rung']} accuracy")
for ax, row in zip(axes[1], mitigation_results):
    ax.bar(["clean", "mitigated"], [row["clean_gap"], row["mitigated_gap"]], color=["tab:orange", "tab:red"])
    ax.set_ylim(0, 1.05)
    ax.set_title(f"D{row['rung']} DP gap")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
xs = [row["rung"] for row in mitigation_results]
ax.plot(xs, [row["clean_gap"] for row in mitigation_results], marker="o", label="clean threshold")
ax.plot(xs, [row["mitigated_gap"] for row in mitigation_results], marker="o", label="mitigated threshold")
ax.set_xlabel("ladder rung")
ax.set_ylabel("demographic-parity gap")
ax.legend()
plt.show()


## Pitfall on D5: tuning $\lambda$ without validation

The wrong workflow picks the lambda that looks fairest on the same slice used for reporting. The fix uses a held-out group-stratified validation slice and prints denominators for the subgroup confusion matrices.

In [ ]:

name, X, y = clf_ladder()[-1]
group = make_group(X, y)
x_train, x_temp, y_train, y_temp, idx_train, idx_temp, g_train, g_temp = safe_split(X, y, group, test_size=0.5)
x_valid, x_test, y_valid, y_test, idx_valid, idx_test, g_valid, g_test = safe_split(x_temp, y_temp, g_temp, test_size=0.5)
scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_valid_s = scaler.transform(x_valid)
x_test_s = scaler.transform(x_test)
model = LogisticRegression(max_iter=2000)
model.fit(x_train_s, y_train)
wrong = []
for lambda_value in [0.0, 0.25, 0.5, 1.0, 2.0]:
    picked = mitigate_bias(model, x_test_s, y_test, g_test, lambda_=lambda_value)
    wrong.append((lambda_value, picked["report"]["dp_gap"], 1.0 - picked["error"]))
right = []
for lambda_value in [0.0, 0.25, 0.5, 1.0, 2.0]:
    picked_valid = mitigate_bias(model, x_valid_s, y_valid, g_valid, lambda_=lambda_value)
    threshold = picked_valid["threshold"]
    scores = model.predict_proba(x_test_s)[:, np.where(model.classes_ == np.max(y_test))[0][0]]
    preds = np.where(scores >= threshold, np.max(y_test), np.min(y_test))
    report = fairness_report(y_test, preds, g_test)
    right.append((lambda_value, report["dp_gap"], accuracy_score(y_test, preds), report))
print("wrong same-data lambda scan", [(a, round(b, 3), round(c, 3)) for a, b, c in wrong])
print("fixed validation lambda scan", [(a, round(b, 3), round(c, 3)) for a, b, c, _ in right])
for group_id in [0, 1]:
    summary = right[2][3][f"group{group_id}"]
    print("group", group_id, "n", summary["n"], "tp", summary["tp"], "fp", summary["fp"], "fn", summary["fn"], "tn", summary["tn"])


## Evaluate it + Practice

- Metric: fairness gap at constrained accuracy.
- No-skill baseline: predict the majority class and report its group selection gap.
- Cheap sanity check: flip group labels and confirm the report changes sign only through the named group rates.
- Ablation: set lambda to zero and watch the gap penalty disappear.
- Failure signals: accuracy rises while subgroup denominators or calibration slices deteriorate.

Practice prompts:
1. Change one stress knob and predict the direction of the metric before running.


2. Replace demographic parity with equalized-odds gap and compare conclusions.

3. Add a group-stratified bootstrap interval around the D5 gap.